In [1]:
# ===== CELL 1: CẤU HÌNH, QUÉT DỮ LIỆU & KHỞI TẠO DATASET (ĐÃ FIX LỖI LANE = 0) =====
import os, time, math, csv, random
from pathlib import Path
import cv2
cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from transformers import SegformerForSemanticSegmentation

CONFIG = {
    "root_dir": r"D:\Dow\BDD100K_UNIFIED_YOLOP",
    "output_dir": r"D:\Dow\outputs_segformer_maxping_v2", 
    "resume_weights": None,
    "pretrained_name": "nvidia/segformer-b2-finetuned-cityscapes-1024-1024",
    "num_classes": 3,

    "img_w": 768,
    "img_h": 432,
    "lane_dilate": 2,

    "epochs": 3,
    "batch_size": 2,
    "grad_accum_steps": 8,       # Effective batch size = 16
    "lr": 6e-5,
    "min_lr": 1e-6,
    "weight_decay": 0.01,
    "warmup_ratio": 0.05,
    "class_weights": [1.0, 1.5, 4.0],

    # CẤU HÌNH TỰ NGHỈ 5 PHÚT RỒI CHẠY TIẾP:
    "steps_per_chunk": 4000,     # Chạy ~34 phút/chặng
    "auto_cooldown_minutes": 5,  # Nghỉ đúng 5 phút rồi tự động train chặng tiếp theo
    "num_workers": 0,
    "seed": 42,
    "max_train_samples": None,
    "max_val_samples": 1000,
}

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
os.makedirs(CONFIG["output_dir"], exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class BDD100KSegmentationDataset(Dataset):
    def __init__(self, root_dir, split="train", img_w=768, img_h=432,
                 lane_dilate=2, is_train=True, max_samples=None):
        self.img_w, self.img_h = img_w, img_h
        self.is_train = is_train
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        self.dilate_kernel = (
            cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (lane_dilate * 2 + 1, lane_dilate * 2 + 1))
            if lane_dilate > 0 else None
        )

        root = Path(root_dir)
        img_dir = root / "images" / split
        drv_dir = root / "labels" / "drivable" / split
        lane_dir = root / "labels" / "lane" / split
        if not drv_dir.exists(): drv_dir = root / "labels" / "drivable"
        if not lane_dir.exists(): lane_dir = root / "labels" / "lane"

        imgs = {p.stem: str(p) for p in img_dir.glob("*.*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]}
        drvs = {p.stem.replace("_drivable_id", "").replace("_drivable_color", "").replace("_drivable", ""): str(p) for p in drv_dir.glob("*.png")}
        lans = {p.stem.replace("_lane_id", "").replace("_lane_color", "").replace("_lane", ""): str(p) for p in lane_dir.glob("*.png")}

        both = sorted(list(set(imgs.keys()) & set(drvs.keys()) & set(lans.keys())))
        self.samples = [(imgs[k], drvs[k], lans[k]) for k in both]

        if max_samples is not None and len(self.samples) > max_samples:
            self.samples = random.Random(42).sample(self.samples, max_samples)
        print(f"✅ [{split.upper()}] Đã nạp {len(self.samples)} mẫu hợp lệ.")

    def __len__(self):
        return len(self.samples)

    def _augment(self, img, mask):
        if random.random() < 0.5:
            img, mask = cv2.flip(img, 1), cv2.flip(mask, 1)

        if random.random() < 0.5:
            scale = random.uniform(0.85, 1.15)
            nh, nw = int(self.img_h * scale), int(self.img_w * scale)
            img_s = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
            mask_s = cv2.resize(mask, (nw, nh), interpolation=cv2.INTER_NEAREST)
            if scale >= 1.0:
                y0, x0 = random.randint(0, nh - self.img_h), random.randint(0, nw - self.img_w)
                img, mask = img_s[y0:y0 + self.img_h, x0:x0 + self.img_w], mask_s[y0:y0 + self.img_h, x0:x0 + self.img_w]
            else:
                pad_img = np.zeros((self.img_h, self.img_w, 3), dtype=img.dtype)
                pad_mask = np.zeros((self.img_h, self.img_w), dtype=mask.dtype)
                y0, x0 = random.randint(0, self.img_h - nh), random.randint(0, self.img_w - nw)
                pad_img[y0:y0 + nh, x0:x0 + nw] = img_s
                pad_mask[y0:y0 + nh, x0:x0 + nw] = mask_s
                img, mask = pad_img, pad_mask

        if random.random() < 0.6:
            img = np.clip(img.astype(np.float32) * random.uniform(0.75, 1.25) + random.uniform(-25, 25), 0, 255).astype(np.uint8)
        return img, mask

    def __getitem__(self, idx):
        ip, dp, lp = self.samples[idx]
        img = cv2.cvtColor(cv2.imread(ip, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        drv = cv2.imread(dp, cv2.IMREAD_GRAYSCALE)
        lane = cv2.imread(lp, cv2.IMREAD_GRAYSCALE)

        img = cv2.resize(img, (self.img_w, self.img_h), interpolation=cv2.INTER_LINEAR)
        drv = cv2.resize(drv, (self.img_w, self.img_h), interpolation=cv2.INTER_NEAREST)
        lane = cv2.resize(lane, (self.img_w, self.img_h), interpolation=cv2.INTER_NEAREST)

        # Nhận diện Drivable
        drv_bin = ((drv == 0) | (drv == 1)) if int(drv[0, 0]) == 2 else ((drv > 0) & (drv != 255))

        # SỬA LỖI NHẬN DIỆN LANE: Tự động phát hiện nền trắng (255) hoặc nền đen (0)
        if lane.max() == 255 and (lane == 255).sum() > (lane == 0).sum():
            lane_bin = (lane < 255).astype(np.uint8)
        else:
            lane_bin = (lane > 0).astype(np.uint8)

        if self.dilate_kernel is not None:
            lane_bin = cv2.dilate(lane_bin, self.dilate_kernel, iterations=1)

        mask = np.zeros((self.img_h, self.img_w), dtype=np.uint8)
        mask[drv_bin] = 1
        mask[lane_bin > 0] = 2

        if self.is_train:
            img, mask = self._augment(img, mask)

        img_f = (img.astype(np.float32) / 255.0 - self.mean) / self.std
        return torch.from_numpy(img_f.transpose(2, 0, 1)).float(), torch.from_numpy(mask.astype(np.int64))

train_ds = BDD100KSegmentationDataset(CONFIG["root_dir"], "train", CONFIG["img_w"], CONFIG["img_h"],
                                      CONFIG["lane_dilate"], True, CONFIG["max_train_samples"])
val_ds = BDD100KSegmentationDataset(CONFIG["root_dir"], "val", CONFIG["img_w"], CONFIG["img_h"],
                                    CONFIG["lane_dilate"], False, CONFIG["max_val_samples"])

# Kiểm tra nhanh tỷ lệ pixel Lane ngay khi khởi tạo để chắc chắn > 0%
_, sample_mask = val_ds[0]
lane_ratio = (sample_mask == 2).float().mean().item() * 100
print(f"🔍 Kiểm tra mẫu ảnh đầu tiên: Tỷ lệ pixel vạch kẻ đường (Lane) = {lane_ratio:.2f}% (Phải > 0%)")

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,
                          num_workers=CONFIG["num_workers"], pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"] * 2, shuffle=False,
                        num_workers=CONFIG["num_workers"], pin_memory=True)

d:\Code\Đồ_án\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1124 08:30:19.371000 23820 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


✅ [TRAIN] Đã nạp 70000 mẫu hợp lệ.
✅ [VAL] Đã nạp 1000 mẫu hợp lệ.
🔍 Kiểm tra mẫu ảnh đầu tiên: Tỷ lệ pixel vạch kẻ đường (Lane) = 1.36% (Phải > 0%)


In [ ]:
# ===== CELL 2: TRAIN SEGFORMER-B2 =====
class HybridSegmentationLoss(nn.Module):
    def __init__(self, class_weights=(1.0, 1.5, 4.0), ce_weight=0.5, dice_weight=0.5, smooth=1.0):
        super().__init__()
        self.ce_weight, self.dice_weight, self.smooth = ce_weight, dice_weight, smooth
        self.register_buffer("weights", torch.tensor(class_weights, dtype=torch.float32))

    def forward(self, logits, targets):
        if logits.shape[-2:] != targets.shape[-2:]:
            logits = F.interpolate(logits, size=targets.shape[-2:], mode="bilinear", align_corners=False)
        ce = F.cross_entropy(logits, targets, weight=self.weights)
        probs = F.softmax(logits, dim=1)
        targets_oh = F.one_hot(targets, num_classes=probs.shape[1]).permute(0, 3, 1, 2).float()
        inter = torch.sum(probs * targets_oh, (0, 2, 3))
        card = torch.sum(probs + targets_oh, (0, 2, 3))
        dice_loss = 1.0 - ((2.0 * inter + self.smooth) / (card + self.smooth)).mean()
        return self.ce_weight * ce + self.dice_weight * dice_loss

@torch.inference_mode()
def evaluate(model, loader, criterion, num_classes=3, use_tta=False):
    model.eval()
    conf_mat = torch.zeros((num_classes, num_classes), dtype=torch.int64, device=device)
    total_loss, n_batches = 0.0, 0

    for imgs, masks in tqdm(loader, desc="Validating", leave=False):
        imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            logits = F.interpolate(model(pixel_values=imgs).logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
            if use_tta:
                logits_f = F.interpolate(model(pixel_values=torch.flip(imgs, dims=[3])).logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
                logits = 0.5 * (logits + torch.flip(logits_f, dims=[3]))
            loss = criterion(logits, masks)

        total_loss += float(loss.item()); n_batches += 1
        preds = torch.argmax(logits, dim=1)
        k = (masks >= 0) & (masks < num_classes)
        conf_mat += torch.bincount(num_classes * masks[k] + preds[k], minlength=num_classes ** 2).reshape(num_classes, num_classes)

    cm = conf_mat.cpu().numpy().astype(np.float64)
    tp, fp, fn = np.diag(cm), cm.sum(axis=0) - np.diag(cm), cm.sum(axis=1) - np.diag(cm)
    iou = tp / np.maximum(1.0, tp + fp + fn)
    dice = (2.0 * tp) / np.maximum(1.0, 2.0 * tp + fp + fn)
    return {
        "loss": total_loss / max(1, n_batches), "pa": float(tp.sum() / np.maximum(1.0, cm.sum())),
        "miou": float(iou.mean()), "dice": float(dice.mean()),
        "iou_bg": float(iou[0]), "iou_drv": float(iou[1]), "iou_lane": float(iou[2]),
        "dice_drv": float(dice[1]), "dice_lane": float(dice[2]),
        "lane_prec": float(tp[2] / np.maximum(1.0, tp[2] + fp[2])),
        "lane_rec": float(tp[2] / np.maximum(1.0, tp[2] + fn[2])),
    }

id2label = {0: "background", 1: "drivable_area", 2: "lane_marking"}
model = SegformerForSemanticSegmentation.from_pretrained(
    CONFIG["pretrained_name"], num_labels=CONFIG["num_classes"],
    id2label=id2label, label2id={v: k for k, v in id2label.items()}, ignore_mismatched_sizes=True
)

last_ckpt_path = os.path.join(CONFIG["output_dir"], "segformer_chunk_last.pth")
best_ckpt_path = os.path.join(CONFIG["output_dir"], "segformer_best.pth")

if not os.path.exists(last_ckpt_path) and CONFIG["resume_weights"] and os.path.exists(CONFIG["resume_weights"]):
    ckpt = torch.load(CONFIG["resume_weights"], map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt)), strict=False)

model.to(device)
criterion = HybridSegmentationLoss(class_weights=CONFIG["class_weights"]).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])

steps_per_epoch = math.ceil(len(train_loader) / CONFIG["grad_accum_steps"])
total_steps = steps_per_epoch * CONFIG["epochs"]
warmup_steps = int(total_steps * CONFIG["warmup_ratio"])

def lr_lambda(step):
    if step < warmup_steps: return float(step + 1) / float(max(1, warmup_steps))
    progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    min_ratio = CONFIG["min_lr"] / CONFIG["lr"]
    return min_ratio + (1.0 - min_ratio) * (0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

csv_path = os.path.join(CONFIG["output_dir"], "training_metrics_maxping.csv")
csv_fields = ["chunk", "epoch", "batch", "loss", "pa", "miou", "dice", "iou_bg", "iou_drv",
              "iou_lane", "dice_drv", "dice_lane", "lane_prec", "lane_rec", "minutes", "vram_gb"]

start_epoch, start_batch, best_miou, chunk_id = 1, 0, 0.0, 1
if os.path.exists(last_ckpt_path):
    ckpt = torch.load(last_ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if ckpt.get("scaler_state_dict") and len(ckpt["scaler_state_dict"]) > 0:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    start_epoch, start_batch = ckpt["epoch"], ckpt["batch_idx"]
    best_miou = ckpt["best_miou"]
    chunk_id = ckpt.get("chunk_id", 1) + 1
    print(f" Tiếp tục Chặng {chunk_id}: Epoch {start_epoch} | Batch {start_batch}/{len(train_loader)} | Best mIoU={best_miou:.4f}")
else:
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=csv_fields).writeheader()

session_stop = False
curr_epoch, curr_batch = start_epoch, start_batch

try:
    for epoch in range(start_epoch, CONFIG["epochs"] + 1):
        if session_stop: break
        curr_epoch = epoch
        model.train()
        optimizer.zero_grad(set_to_none=True)
        t0, running_loss, step_in_chunk = time.time(), 0.0, 0
        pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Ep {epoch}/{CONFIG['epochs']} (Chặng {chunk_id})")

        for i, (imgs, masks) in pbar:
            if epoch == start_epoch and i < start_batch:
                continue
            curr_batch = i

            imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                logits = model(pixel_values=imgs).logits
                loss = criterion(logits, masks) / CONFIG["grad_accum_steps"]

            scaler.scale(loss).backward()
            running_loss += float(loss.item()) * CONFIG["grad_accum_steps"]
            step_in_chunk += 1

            if (i + 1) % CONFIG["grad_accum_steps"] == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scale_before <= scaler.get_scale():
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            if step_in_chunk % 50 == 0:
                pbar.set_postfix(loss=f"{running_loss / step_in_chunk:.4f}", chunk=f"{step_in_chunk}/{CONFIG['steps_per_chunk']}")

            # KẾT THÚC CHẶNG HOẶC HẾT EPOCH
            if step_in_chunk >= CONFIG["steps_per_chunk"] or (i + 1) == len(train_loader):
                elapsed_min = (time.time() - t0) / 60.0
                vram_gb = (torch.cuda.max_memory_allocated(device) / (1024 ** 3)) if device.type == "cuda" else 0.0
                metrics = evaluate(model, val_loader, criterion, num_classes=CONFIG["num_classes"], use_tta=False)

                print(f"\n[Chặng {chunk_id} | Ep {epoch} Batch {i+1}] Loss={running_loss/step_in_chunk:.4f} | "
                      f"PA={metrics['pa']:.4f} | mIoU={metrics['miou']:.4f} | Dice={metrics['dice']:.4f} | "
                      f"IoU bg/drv/lane={metrics['iou_bg']:.3f}/{metrics['iou_drv']:.3f}/{metrics['iou_lane']:.3f} | "
                      f"Lane P/R={metrics['lane_prec']:.3f}/{metrics['lane_rec']:.3f} | {elapsed_min:.1f}p | VRAM {vram_gb:.2f}GB")

                row = {"chunk": chunk_id, "epoch": epoch, "batch": i + 1, "minutes": round(elapsed_min, 2),
                       "vram_gb": round(vram_gb, 2), **{k: round(v, 4) for k, v in metrics.items()}}
                with open(csv_path, "a", newline="", encoding="utf-8") as f:
                    csv.DictWriter(f, fieldnames=csv_fields).writerow(row)

                next_b = 0 if (i + 1) >= len(train_loader) else (i + 1)
                next_ep = (epoch + 1) if (i + 1) >= len(train_loader) else epoch
                torch.save({
                    "epoch": next_ep, "batch_idx": next_b, "chunk_id": chunk_id,
                    "model_state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(), "scaler_state_dict": scaler.state_dict(),
                    "best_miou": best_miou
                }, last_ckpt_path)

                if metrics["miou"] > best_miou:
                    best_miou = metrics["miou"]
                    torch.save({"model_state_dict": model.state_dict(), "epoch": epoch, "val_miou": best_miou,
                                "metrics": metrics, "config": CONFIG}, best_ckpt_path)
                    model.config.save_pretrained(CONFIG["output_dir"])
                    print(f"   ✅ LƯU KỶ LỤC MỚI segformer_best.pth (mIoU={best_miou:.4f})")

                # TỰ ĐỘNG NGHỈ 5 PHÚT RỒI CHẠY TIẾP CHẶNG SAU
                if step_in_chunk >= CONFIG["steps_per_chunk"] and (i + 1) < len(train_loader):
                    cooldown_min = CONFIG["auto_cooldown_minutes"]
                    if cooldown_min > 0:
                        print(f"💤 GPU đang nghỉ {cooldown_min} phút để hạ nhiệt...")
                        torch.cuda.empty_cache()
                        for m_left in range(cooldown_min, 0, -1):
                            print(f"   ⏳ Còn {m_left} phút nữa sẽ tự động tiếp tục Chặng {chunk_id + 1}...", end="\r")
                            time.sleep(60)
                        print(f"\n Bắt đầu Chặng {chunk_id + 1}...")
                        model.train()
                        step_in_chunk, running_loss, t0 = 0, 0.0, time.time()
                        chunk_id += 1
                    else:
                        print(f" Hoàn thành Chặng {chunk_id} !")
                        session_stop = True; break
        start_batch = 0

except KeyboardInterrupt:
    print(f"\n Dừng thủ công! Đang lưu khẩn cấp tại Epoch {curr_epoch}, Batch {curr_batch}...")
    torch.save({
        "epoch": curr_epoch, "batch_idx": curr_batch, "chunk_id": chunk_id,
        "model_state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(), "scaler_state_dict": scaler.state_dict(),
        "best_miou": best_miou
    }, last_ckpt_path)
    print(f" Đã lưu an toàn tại {last_ckpt_path}!")

if not session_stop and curr_epoch >= CONFIG["epochs"] and os.path.exists(best_ckpt_path):
    print("\n=== ĐÁNH GIÁ CUỐI CÙNG VỚI TTA (ORIGINAL + FLIP) TRÊN BEST CHECKPOINT ===")
    best_ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt["model_state_dict"])
    tta = evaluate(model, val_loader, criterion, num_classes=CONFIG["num_classes"], use_tta=True)
    print(f"[FINAL TTA] PA={tta['pa']:.4f} | mIoU={tta['miou']:.4f} | Dice={tta['dice']:.4f} | "
          f"IoU_Drv={tta['iou_drv']:.4f} | IoU_Lane={tta['iou_lane']:.4f} | "
          f"Lane Prec={tta['lane_prec']:.4f} | Lane Rec={tta['lane_rec']:.4f}")

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `19`.
Loading weights: 100%|██████████| 380/380 [00:00<00:00, 56040.07it/s]
[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b2-finetuned-cityscapes-1024-1024
Key                           | Status   |                                                                                                    
------------------------------+----------+----------------------------------------------------------------------------------------------------
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([19]) vs model:torch.Size([3])                      
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([19, 768, 1, 1]) vs model:torch.Size([3, 768, 1, 1])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


▶️ Tiếp tục Chặng 7: Epoch 1 | Batch 20618/35000 | Best mIoU=0.7160


Ep 1/3 (Chặng 7):  70%|███████   | 24617/35000 [1:06:04<1:27:58,  1.97it/s, chunk=4000/4000, loss=0.1753]


[Chặng 7 | Ep 1 Batch 24618] Loss=0.1753 | PA=0.9541 | mIoU=0.7153 | Dice=0.8057 | IoU bg/drv/lane=0.959/0.821/0.367 | Lane P/R=0.398/0.825 | 66.1p | VRAM 2.73GB
💤 GPU đang nghỉ 5 phút để hạ nhiệt...


Ep 1/3 (Chặng 7):  70%|███████   | 24618/35000 [1:16:04<520:39:47, 180.54s/it, chunk=4000/4000, loss=0.1753]


 Bắt đầu Chặng 8...


... (10 outputs left out)

Ep 1/3 (Chặng 7): 100%|██████████| 35000/35000 [3:13:06<00:00,  3.02it/s, chunk=2350/4000, loss=0.1648] 


   ✅ LƯU KỶ LỤC MỚI segformer_best.pth (mIoU=0.7194)


Ep 2/3 (Chặng 10):  11%|█▏        | 3999/35000 [34:05<4:18:53,  2.00it/s, chunk=4000/4000, loss=0.1585]


[Chặng 10 | Ep 2 Batch 4000] Loss=0.1585 | PA=0.9576 | mIoU=0.7257 | Dice=0.8144 | IoU bg/drv/lane=0.962/0.831/0.384 | Lane P/R=0.416/0.831 | 34.1p | VRAM 3.14GB
   ✅ LƯU KỶ LỤC MỚI segformer_best.pth (mIoU=0.7257)
💤 GPU đang nghỉ 5 phút để hạ nhiệt...
